# Read the Target Sequence / Profile

In [1]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

# Parse the sequences

In [2]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

# Initialize HMM

In [3]:
from modules.model import (
    init_transition_mat,
    remove_silent_states,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=1)

# remove silent D states
cols = list(trans_matdf.columns)
d_cols = [i for i in cols if i[0] == 'D']
trans_mat, red_cols = remove_silent_states(trans_matdf.to_numpy(), cols, d_cols)
trans_matdf = pd.DataFrame(trans_mat, columns=red_cols, index=red_cols)

# Encode HMM as PHMM (hmmlearn)

In [4]:
import numpy as np

A = trans_mat.T
B = emit_matdf.to_numpy().T
startprob = np.zeros(A.shape[0])
startprob[0] = 1.0                           # every path starts in M0

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

# Train HMM

In [5]:
from modules.train import train_phmm
from modules.evaluation import description_accuracy
from modules.model import build_hmmlearn_model

from pathlib import Path
import logging
import joblib

# Create directory for saved models
model_dir = Path("saved_models/phmm")
model_dir.mkdir(parents=True, exist_ok=True)

da0 = description_accuracy(
    build_hmmlearn_model(A, B, startprob),
    sym_idx,
    test_seqs,
)

results = {"untrained": (da0.mean(), da0.std())}
histories = {}

settings = {
    "weak prior (1, 5)": (1.0, 5.0),
    "default (5, 20)": (5.0, 20.0),
    "strong prior (20, 80)": (20.0, 80.0),
}

max_iter = 15
tol = 1e-3

logging.getLogger("hmmlearn").setLevel(logging.ERROR)

for label, (ts, es) in settings.items():
    print(f"--- {label}: trans_strength={ts}, emit_strength={es}")

    model, hist = train_phmm(
        build_hmmlearn_model(
            A, B, startprob,
            trans_strength=ts,
            emit_strength=es,
        ),
        sym_idx,
        train_seqs,
        val_seqs,
        test_seqs,
        max_iter,
        tol,
    )

    da = description_accuracy(model, sym_idx, test_seqs)

    results[label] = (da.mean(), da.std())
    histories[label] = hist

    # Save trained model
    model_path = model_dir / f"{name}_ts{ts:g}_es{es:g}.joblib"
    joblib.dump(model, model_path)

    print(f"Saved model to {model_path}")


--- weak prior (1, 5): trans_strength=1.0, emit_strength=5.0
iter   0  cpu     0.0s  log-lik -14646.97  train_da_mean 0.0025  val_da_mean 0.0014
iter   1  cpu     0.2s  log-lik -13270.52  train_da_mean 0.1000  val_da_mean 0.0858
iter   2  cpu     0.3s  log-lik -12503.14  train_da_mean 0.1573  val_da_mean 0.1311
iter   3  cpu     0.4s  log-lik -11521.20  train_da_mean 0.2305  val_da_mean 0.1948
iter   4  cpu     0.6s  log-lik -10979.94  train_da_mean 0.2694  val_da_mean 0.2251
iter   5  cpu     0.7s  log-lik -10700.63  train_da_mean 0.2881  val_da_mean 0.2405
iter   6  cpu     0.8s  log-lik -10579.59  train_da_mean 0.2959  val_da_mean 0.2465
iter   7  cpu     0.9s  log-lik -10540.33  train_da_mean 0.2985  val_da_mean 0.2482
iter   8  cpu     1.0s  log-lik -10521.77  train_da_mean 0.2998  val_da_mean 0.2487
iter   9  cpu     1.1s  log-lik -10511.06  train_da_mean 0.3005  val_da_mean 0.2487
iter  10  cpu     1.3s  log-lik -10504.86  train_da_mean 0.3008  val_da_mean 0.2485
iter  11  cpu  

# Visualize the Results

In [7]:
from modules.evaluation import (
    plot_da_learning_curve,
    plot_da_bars,
)
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-darkgrid")   # close to the paper's look
except OSError:
    pass

results_dir = Path("results/phmm")

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_training_histories.csv",
    index=False,
)

for label, h in histories.items():
    h.to_csv(results_dir / f"{name}_{label.split()[0]}.csv", index=False)
pd.DataFrame(results, index=["test_da_mean", "test_da_std"]).T.to_csv(results_dir / f"{name}_da_results.csv")

references = {}
plot_da_learning_curve(histories, references, out=results_dir / f"{name}_da_vs_time.png")
plot_da_learning_curve(histories, references, x='iter', out=results_dir / f"{name}_da_vs_epoch.png")
plot_da_bars(results, references, out=results_dir / f"{name}_da_bars.png")